In [4]:
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

print("CUDA:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


torch: 2.10.0+cu128
transformers: 4.36.2
datasets: 2.16.1
accelerate: 0.25.0
CUDA: True


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [5]:
from datasets import load_dataset

dataset = load_dataset("abnerh/TORGO-database")
dataset = dataset["train"]

print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'audio': {'path': 'FC01_1_arrayMic_0066.wav', 'array': array([ 0.00125122,  0.00387573,  0.00115967, ...,  0.00149536,
       -0.00326538,  0.00027466]), 'sampling_rate': 16000}, 'transcription': 'alpha', 'speech_status': 'healthy', 'gender': 'female', 'duration': 3.3}


In [6]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "openai/whisper-base"

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
def transcribe(batch):
    audio = batch["audio"]

    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(inputs.input_features)

    transcription = processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0]

    return {"prediction": transcription}

In [8]:
small_ds = dataset.select(range(100))  # baseline sample
results = small_ds.map(transcribe)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [9]:
from jiwer import wer

refs = results["transcription"]
preds = results["prediction"]

print("WER:", wer(refs, preds))

WER: 0.509009009009009


In [10]:
dataset = load_dataset("abnerh/TORGO-database", split="train")
dataset = dataset.train_test_split(test_size=0.1)

test_ds = dataset["test"]
results = test_ds.map(transcribe)

Map:   0%|          | 0/1656 [00:00<?, ? examples/s]

In [13]:
from jiwer import wer

refs = results["transcription"]
preds = results["prediction"]

print("WER:", wer(refs, preds))

WER: 0.5629186602870814


In [14]:
# Noramlize for better WER

from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip

transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

refs_clean = [transform(r) for r in refs]
preds_clean = [transform(p) for p in preds]

print("Normalized WER:", wer(refs_clean, preds_clean))

Normalized WER: 0.3215311004784689


In [15]:
# Seperate dysarthia and non-dysarthia

dys_refs, dys_preds = [], []
healthy_refs, healthy_preds = [], []

for r in results:
    if r["speech_status"] == "dysarthria":
        dys_refs.append(r["transcription"])
        dys_preds.append(r["prediction"])
    else:
        healthy_refs.append(r["transcription"])
        healthy_preds.append(r["prediction"])

print("Dysarthria WER:", wer(dys_refs, dys_preds))
print("Healthy WER:", wer(healthy_refs, healthy_preds))

Dysarthria WER: 0.7563527653213752
Healthy WER: 0.47185080928923295


In [16]:
from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip

# normalization pipeline
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

# --- overall normalized ---
refs_clean = [transform(r) for r in refs]
preds_clean = [transform(p) for p in preds]

print("Normalized Overall WER:", wer(refs_clean, preds_clean))


# --- split into groups ---
dys_refs, dys_preds = [], []
healthy_refs, healthy_preds = [], []

for r in results:
    if r["speech_status"] == "dysarthria":
        dys_refs.append(r["transcription"])
        dys_preds.append(r["prediction"])
    else:
        healthy_refs.append(r["transcription"])
        healthy_preds.append(r["prediction"])


# --- normalize splits ---
dys_refs_clean = [transform(r) for r in dys_refs]
dys_preds_clean = [transform(p) for p in dys_preds]

healthy_refs_clean = [transform(r) for r in healthy_refs]
healthy_preds_clean = [transform(p) for p in healthy_preds]


# --- compute WER ---
print("Normalized Dysarthria WER:", wer(dys_refs_clean, dys_preds_clean))
print("Normalized Healthy WER:", wer(healthy_refs_clean, healthy_preds_clean))

Normalized Overall WER: 0.3215311004784689
Normalized Dysarthria WER: 0.5994020926756353
Normalized Healthy WER: 0.19071076706544687
